# Sylvester's AI Lab — Cloud Render (Google Colab)
Runs the **full free pipeline** on Google's free compute: script → voiceover → Remotion render (batched) → YouTube upload → Telegram.

Your phone can't render Remotion (headless Chromium stalls on-device), but a cloud machine renders 1080p in minutes.

## Steps
1. Run the **Install** cell.
2. **Get the project**: paste your GitHub repo URL, OR upload `ai-lab-internal.zip` (zipped from your phone).
3. **Secrets** (🔑 *Secrets* tab, top-left): `GEMINI_API_KEY`, `TELEGRAM_BOT_TOKEN`, `YOUTUBE_API_KEY`. Then upload your `client_secrets.json` (YouTube OAuth Desktop client) when prompted.
4. Run the **pipeline** cell → it generates the script, voiceover, and a **60-second preview**. Download the preview (left file browser) and judge the quality.
5. Run the **full render** cell → renders the whole episode in resumable batches, uploads it (private) and pings Telegram.

> On first upload you'll get a YouTube consent URL — open it in your browser, authorize, paste the code back.

In [ ]:
import subprocess, sys, os, shutil

# --- Node (Colab ships a recent Node; verify) ---
print("node:", subprocess.run(["node","--version"], capture_output=True, text=True).stdout.strip())

# --- Chromium for headless Remotion ---
if not (shutil.which("chromium") or shutil.which("chromium-browser")):
    subprocess.run("apt-get update -qq && apt-get install -y -qq chromium >/dev/null 2>&1", shell=True)
print("chromium:", shutil.which("chromium") or shutil.which("chromium-browser"))

# --- Python deps ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "edge-tts", "google-api-python-client", "google-auth-oauthlib",
                "google-auth", "python-dotenv"], check=True)
print("pip deps ok")

In [ ]:
from google.colab import files
import os, subprocess

REPO_URL = ""  # <-- paste your GitHub repo URL, e.g. "https://github.com/you/sylvesters-ai-lab.git"
PROJECT = "/content/ai-lab-internal"

if REPO_URL:
    subprocess.run(f"git clone {REPO_URL} {PROJECT}", shell=True, check=True)
    print("cloned ->", PROJECT)
else:
    print("No REPO_URL set. Upload ai-lab-internal.zip (the folder zipped from your phone):")
    uploaded = files.upload()  # select ai-lab-internal.zip
    zipname = next(iter(uploaded))
    subprocess.run(f"unzip -q -o '{zipname}' -d /content/", shell=True)
    print("unzipped; project at", PROJECT, "->", os.path.isdir(PROJECT))

In [ ]:
import os, shutil
from google.colab import userdata, files

# Point config's ~/ai-lab-internal paths at /content/ai-lab-internal
os.environ["HOME"] = "/content"
PROJECT = "/content/ai-lab-internal"
os.chdir(PROJECT)

def sec(k, d=""):
    try:
        return userdata.get(k)
    except Exception:
        return os.getenv(k, d)

os.environ["GEMINI_API_KEY"] = sec("GEMINI_API_KEY")
os.environ["TELEGRAM_BOT_TOKEN"] = sec("TELEGRAM_BOT_TOKEN")
os.environ["YOUTUBE_API_KEY"] = sec("YOUTUBE_API_KEY")
os.environ["TELEGRAM_CHAT_ID"] = sec("TELEGRAM_CHAT_ID", "8800205878")

# Chromium path for headless Remotion
chrome = shutil.which("chromium") or shutil.which("chromium-browser")
os.environ["REMOTION_BROWSER_EXECUTABLE"] = chrome or ""
print("browser:", os.environ["REMOTION_BROWSER_EXECUTABLE"])

print("Upload client_secrets.json (YouTube OAuth *Desktop* client):")
files.upload()  # lands in CWD = /content/ai-lab-internal
print("client_secrets present:", os.path.exists(os.path.join(PROJECT, "client_secrets.json")))

In [ ]:
%cd /content/ai-lab-internal
!npm install 2>&1 | tail -3
!npm install framer-motion 2>&1 | tail -3
print("npm install done")

In [ ]:
import os, sys, json
from pathlib import Path

os.environ["HOME"] = "/content"          # resolve config paths to /content/ai-lab-internal
sys.path.insert(0, "/content/ai-lab-internal/pipeline")

import config
import script_generator, voiceover, render_trigger

TOPIC = "GPT-5.6-Sol: Is This New AI Model Actually Worth the Switch?"
slug = "".join(c if c.isalnum() else "-" for c in TOPIC.lower()).strip("-")
ws = config.WORKSPACE / slug
ws.mkdir(parents=True, exist_ok=True)
audio = ws / config.VOICEOVER_FILENAME

# 1. Script
if not (ws / config.SCRIPT_JSON_REL).exists():
    script = script_generator.generate_script(TOPIC, ws)
else:
    script = json.loads((ws / config.SCRIPT_JSON_REL).read_text())

# 2. Voiceover (Edge TTS)
if not audio.exists():
    voiceover.generate_voiceover(script, audio)

# 3. PREVIEW — first 60s, with audio. Download & judge quality.
preview = render_trigger.render_preview(slug, audio, seconds=60)
print("\nPREVIEW READY:", preview)
print("-> Download it from the left file browser, judge the quality,")
print("   then run the next cell for the full episode + upload.")

In [ ]:
import os, sys, json
from pathlib import Path

os.environ["HOME"] = "/content"
sys.path.insert(0, "/content/ai-lab-internal/pipeline")

import config, render_trigger, uploader, notifier

TOPIC = "GPT-5.6-Sol: Is This New AI Model Actually Worth the Switch?"
slug = "".join(c if c.isalnum() else "-" for c in TOPIC.lower()).strip("-")
ws = config.WORKSPACE / slug
audio = ws / config.VOICEOVER_FILENAME
script = json.loads((ws / config.SCRIPT_JSON_REL).read_text())

# Full episode in resumable batches, then concat + mux.
full = render_trigger.render_batches(slug, audio)
print("FULL VIDEO:", full)

# Upload (private) — first run prints a YouTube consent URL; paste the code.
vid = uploader.upload(full,
    title=script.get("title", TOPIC),
    description=script.get("description", ""),
    tags=script.get("tags", []),
    privacy="private")
url = f"https://www.youtube.com/watch?v={vid}"
notifier.send_notification(f"Uploaded *private*: [{script.get('title', TOPIC)}]({url})")
print("\nDONE:", url)